In [12]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [26]:
from curl_cffi import requests
from bs4 import BeautifulSoup

url = 'https://www.ambitionbox.com/list-of-companies?page=1'

# Note: pass url without quotes, and use impersonate="chrome120" to bypass bot protection
webpage = requests.get(url, impersonate="chrome120")

# Parse the HTML content
soup = BeautifulSoup(webpage.text, 'html.parser')

# Check title to verify it loaded correctly
print("Page Title:", soup.title.text if soup.title else "No title found")


Page Title: Top Companies in India | AmbitionBox


In [62]:
company = soup.find_all('div',class_='companyCardWrapper')
len(company)

20

In [ ]:
name = []
ratings =[]
disc = []
high_rated = []
critical_rated = []
reviews = []
salaries =[]
interviews = []
jobs = []
benefits = []


for i in company:
  
    name.append(i.find('h2').text.strip())
    ratings.append(i.find('div',class_='rating_text').text.strip())
    disc.append(i.find('span',class_='companyCardWrapper__interLinking').text.strip())

    rating_values = i.find_all('span',class_='companyCardWrapper__ratingValues')
    if len(rating_values) >= 2:
        high_rated.append(rating_values[0].get_text(strip=True))
        critical_rated.append(rating_values[1].get_text(strip=True))
    else:
        high_rated.append(None)
        critical_rated.append(None)


    data = i.find_all('span',class_='companyCardWrapper__ActionCount') 
    reviews.append(data[0].get_text(strip=True)) 
    salaries.append(data[1].get_text(strip=True))
    interviews.append(data[2].get_text(strip=True))
    jobs.append(data[3].get_text(strip=True))
    benefits.append(data[4].get_text(strip=True))




['11k', '7k', '4.9k', '5.7k', '3.9k', '3.4k', '4.9k', '4k', '3.8k', '3.5k', '3.7k', '2k', '2.2k', '2.6k', '3.2k', '3.5k', '1.9k', '925', '895', '1.4k']


#creating the DataFrame 

In [85]:
d = { 'Name':name,
    'ratings':ratings,
    'Discription':disc,
    'Highly Rated for':high_rated,
    'Critical Rated for':critical_rated,
    'Reviews':reviews,
    'Salaries':salaries,
    'Interviews':interviews,
    'Jobs':jobs,
    'Benefits':benefits
}

df = pd.DataFrame(d)

In [86]:
df

,Name,ratings,Discription,Highly Rated for,Critical Rated for,Reviews,Salaries,Interviews,Jobs,Benefits
0,TCS,3.2,IT Services & Consulting | Bengaluru +478 othe...,Job Security,"Promotions, Salary, Work Satisfaction",1.2L,10.4L,11.4k,5.6k,11k
1,Accenture,3.7,IT Services & Consulting | Bengaluru +280 othe...,NaN,NaN,76.3k,7.3L,9.6k,16.6k,7k
2,Wipro,3.6,IT Services & Consulting | Hyderabad +392 othe...,NaN,NaN,67.2k,4.9L,7k,3,4.9k
3,Cognizant,3.7,IT Services & Consulting | Hyderabad +254 othe...,NaN,NaN,63.5k,6.1L,6.6k,961,5.7k
4,Capgemini,3.6,IT Services & Consulting | Bengaluru +204 othe...,"Work Life Balance, Job Security","Promotions, Salary, Work Satisfaction",55.7k,5L,5.7k,2.2k,3.9k
5,HDFC Bank,3.8,Banking | Mumbai +1904 other locations,Job Security,"Promotions, Salary",54.9k,1.6L,3.2k,525,3.4k
6,Infosys,3.5,IT Services & Consulting | Bengaluru +258 othe...,Job Security,"Promotions, Salary, Work Satisfaction",50.5k,5.4L,8.6k,3.8k,4.9k
7,HCLTech,3.4,IT Services & Consulting | Bengaluru +363 othe...,NaN,NaN,48.4k,4L,4.7k,326,4k
8,ICICI Bank,4.0,Banking | Mumbai +1476 other locations,NaN,NaN,47.2k,1.6L,3k,23,3.8k
9,Tech Mahindra,3.3,IT Services & Consulting | Hyderabad +339 othe...,NaN,NaN,44.9k,2.9L,4.7k,823,3.5k


In [ ]:
final = pd.DataFrame()

# Scrape pages 1 to 499
for j in range(1, 500):

    print("Scraping page:", j)

    # URL
    url = 'https://www.ambitionbox.com/list-of-companies?page={}'.format(j)

    # Request webpage
    webpage = requests.get(
        url,
        impersonate="chrome120"
    )

    # Create BeautifulSoup object
    soup = BeautifulSoup(webpage.text, 'html.parser')

    # Find all company cards
    company = soup.find_all(
        'div',
        class_='companyCardWrapper'
    )

    # Stop if no companies are found
    if not company:
        print("No companies found. Stopping...")
        break

    # Empty lists for current page
    name = []
    ratings = []
    disc = []
    high_rated = []
    critical_rated = []
    reviews = []
    salaries = []
    interviews = []
    jobs = []
    benefits = []

    # Extract data from each company
    for i in company:

        
        # Company Name
        
        name_tag = i.find('h2')

        name.append(
            name_tag.get_text(strip=True)
            if name_tag else None
        )

        
        # Rating
        
        rating_tag = i.find(
            'div',
            class_='rating_text'
        )

        ratings.append(
            rating_tag.get_text(strip=True)
            if rating_tag else None
        )

        
        # Description
        
        disc_tag = i.find(
            'span',
            class_='companyCardWrapper__interLinking'
        )

        disc.append(
            disc_tag.get_text(strip=True)
            if disc_tag else None
        )

        
        # Highly Rated / Critical Rated
        
        rating_values = i.find_all(
            'span',
            class_='companyCardWrapper__ratingValues'
        )

        if len(rating_values) >= 2:

            high_rated.append(
                rating_values[0].get_text(strip=True)
            )

            critical_rated.append(
                rating_values[1].get_text(strip=True)
            )

        elif len(rating_values) == 1:

            high_rated.append(
                rating_values[0].get_text(strip=True)
            )

            critical_rated.append(None)

        else:

            high_rated.append(None)
            critical_rated.append(None)

        # Reviews, Salaries,
        # Interviews, Jobs, Benefits
      
        data = i.find_all(
            'span',
            class_='companyCardWrapper__ActionCount'
        )

        values = [
            x.get_text(strip=True)
            for x in data
        ]

        reviews.append(
            values[0] if len(values) > 0 else None
        )

        salaries.append(
            values[1] if len(values) > 1 else None
        )

        interviews.append(
            values[2] if len(values) > 2 else None
        )

        jobs.append(
            values[3] if len(values) > 3 else None
        )

        benefits.append(
            values[4] if len(values) > 4 else None
        )

   
    # Create DataFrame
 
    df = pd.DataFrame({

        'Name': name,

        'Ratings': ratings,

        'Description': disc,

        'Highly Rated for': high_rated,

        'Critical Rated for': critical_rated,

        'Reviews': reviews,

        'Salaries': salaries,

        'Interviews': interviews,

        'Jobs': jobs,

        'Benefits': benefits
    })


    final = pd.concat(
        [final, df],
        ignore_index=True
    )



print("\nScraping completed!")

print("Total companies scraped:", len(final))

Scraping page: 1
Scraping page: 2
Scraping page: 3
Scraping page: 4
Scraping page: 5
Scraping page: 6
Scraping page: 7
Scraping page: 8
Scraping page: 9
Scraping page: 10
Scraping page: 11
Scraping page: 12
No companies found. Stopping...

Scraping completed!
Total companies scraped: 220


In [102]:
final.to_csv('ambitionbox_companies.csv', index=False)